# 第 20 课｜第一次装入 MaleCNS 子图：先验证 image，再谈结果

项目最终需要让 software 与 FPGA 消费**同一份**转换后的 network image。

今天只问一个问题：

> **怎样证明两个 consumer 收到的是同一版本、完全相同的 network bytes？**

本课主要新概念：**manifest + checksum 组成的 integrity contract。**

重要边界：正式的 RMD-017/018 MaleCNS artifact 目前**尚未声明完成**。本课用极小 teaching fixture 学习 loading contract；**不声称这份 fixture 就是真实 MaleCNS 数据**。

## 1. 概念账本

**已经知道：** connectome node/edge/metadata、binary data movement、host control，以及系统的 **现场可编程门阵列（Field-Programmable Gate Array, FPGA）** 一侧。

**今天学习：** **manifest**（描述 artifact 的小型清单）与 **checksum**（对精确 bytes 计算的确定性摘要）。本课具体采用 **256 位安全散列算法（Secure Hash Algorithm 256-bit, SHA-256）** 作为 checksum algorithm。

**只预告：** 真实 MaleCNS converter、正式 binary schema、1K differential test 与完整数据装载。

## 2. 哪些东西必须一起走？

一个可重复的 network artifact 不能只有文件名。至少要有：

- schema/version identifier；
- byte length；
- 精确 image bytes 的 checksum；
- 足够的 provenance，说明来自哪个 converter / data release。

checksum 回答的是：**“这两份 bytes 是否完全相同？”**  
它不回答：**“这是不是科学上正确的 network？”**

## 3. conversion 与 replay

<div style="max-width:860px; margin:1rem auto;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 860 420" role="img" aria-label="versioned network image integrity flow" style="width:100%; height:auto; display:block;">
  <defs>
    <marker id="l20-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#2f5f3f"/>
    </marker>
  </defs>
  <g font-family="sans-serif" font-size="21" text-anchor="middle">
    <rect x="35" y="155" width="190" height="78" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="130" y="187" fill="#1f2d24">source data</text><text x="130" y="214" fill="#1f2d24">+ release ID</text>
    <rect x="285" y="155" width="190" height="78" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="380" y="187" fill="#1f2d24">versioned</text><text x="380" y="214" fill="#1f2d24">converter</text>
    <rect x="535" y="60" width="240" height="78" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="655" y="108" fill="#1f2d24">binary image</text>
    <rect x="535" y="250" width="240" height="92" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="655" y="282" fill="#1f2d24">manifest</text><text x="655" y="309" fill="#1f2d24">version + provenance</text><text x="655" y="336" fill="#1f2d24">byte count + SHA-256</text>
    <rect x="650" y="155" width="160" height="58" rx="8" fill="#f4fbf6" stroke="#3f7a50" stroke-width="2"/>
    <text x="730" y="191" fill="#1f2d24">consumers</text>
  </g>
  <path d="M225 194 L285 194" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l20-arrow)"/>
  <path d="M475 178 C500 150,520 125,548 112" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l20-arrow)"/>
  <path d="M475 210 C505 235,520 260,548 278" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l20-arrow)"/>
  <path d="M655 138 L700 155" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l20-arrow)"/>
  <path d="M655 250 L700 213" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l20-arrow)"/>
</svg>
</div>

## 4. Run：制作并验证 teaching image

下面的 payload 只是为了让 integrity workflow 在离线环境可运行而准备的 byte fixture，不是正式 MaleCNS release artifact。

In [ ]:
import hashlib
import json

payload = bytes([19, 20, 1, 0, 3, 2, 7, 11])
manifest = {
    "schema_version": "teaching-v1",
    "byte_count": len(payload),
    "sha256": hashlib.sha256(payload).hexdigest(),
}

software_bytes = bytes(payload)
fpga_replay_bytes = bytes(payload)

print(json.dumps(manifest, indent=2))
print("software checksum:", hashlib.sha256(software_bytes).hexdigest())
print("FPGA replay checksum:", hashlib.sha256(fpga_replay_bytes).hexdigest())
print("same image:", software_bytes == fpga_replay_bytes)

## 5. Observe

两个 consumer 收到完全相同的 bytes，因此 SHA-256 checksum 一致。

这只是 **integrity proof**，还不是 neural correctness proof。后续 T-015 才比较真实 subset 在 software 与 FPGA 上产生的行为。

## 6. 为什么 converter 也要 version？

如果 conversion rule 改变——例如 edge ordering、weight encoding、record width 改了——同一份 scientific source dataset 也可能变成不同的 binary bytes。

所以 manifest 中要有 converter/schema version。可重复性需要记录**数据版本 + 转换规则**，而不只是 source 名字。

## 7. integrity 之后才做 differential test

比较干净的顺序是：

1. 先验证 manifest 与 checksum；
2. 两个 implementation 装入同一 image；
3. replay 同一 initial state 与 input events；
4. 用批准的 oracle 比较 output/state。

如果第 1 步都失败，后面的行为差异就很难解释。

## 8. Try It

修改 payload 中一个 byte，再计算 digest。先预测 manifest 中哪些字段会改变。

不同 checksum 能告诉你**为什么**这个 byte 变化了吗？

## 9. 作业

[第 20 课作业：建立最小 image manifest](../../exercises/zh/20_load_malecns_subset.ipynb)

## 10. AI Task

让 AI 提议一个 manifest schema。把每个 field 标成 integrity、provenance、schema/version 或 experiment state。拒绝任何把 runtime neuron state 偷偷混进 static connectome image 的设计。

## 11. Human Check

解释为什么 checksum 相同是正确结果的必要条件，却不是充分条件。为什么 software 与 FPGA differential test 必须消费精确相同的 binary image？

## 12. Engineering Handoff

对应 `RMD-017 / RMD-018`、`MOD-011` 与 `T-014 / T-015`。正式工程 slice 必须用文档化的 MaleCNS-derived artifact 替换 teaching fixture，之后才能宣称 real-subset result。

## 13. Project Trace

- Lesson：`LSN-020`
- 映射：`RMD-018`，前置 `RMD-017`
- 需求路径：`TRACE-C-001`
- integrity oracle：`T-014`
- real-subset behavioral oracle：`T-015`

## 14. Exit Ticket

面对 binary image 与 manifest，你能够验证 byte count 与 checksum，解释这证明了什么，并说明还需要怎样的 differential behavioral test。